<a href="https://colab.research.google.com/github/Samarth101/ML_Research/blob/main/enhance_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import shutil

In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels):
        super(ResidualBlock,self).__init__()
        self.block=nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(in_channels,in_channels,kernel_size=3),
            nn.InstanceNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(in_channels, in_channels, kernel_size=3),
            nn.InstanceNorm2d(in_channels)
        )
    def forward(self,x):
        return x + self.block(x)

In [6]:
class GeneratorResNet(nn.Module):
    def __init__(self, input_shape=(3, 256, 256), num_residual_blocks=9):
        super(GeneratorResNet, self).__init__()
        channels = input_shape[0]
        out_features = 64
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(channels, out_features, kernel_size=7),
            nn.InstanceNorm2d(out_features),
            nn.ReLU(inplace=True)
        ]
        in_features = out_features
        for _ in range(2):
            out_features *= 2
            model += [
                nn.Conv2d(in_features, out_features, kernel_size=3, stride=2, padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
        for _ in range(num_residual_blocks):
            model += [ResidualBlock(out_features)]
        for _ in range(2):
            out_features //= 2
            model += [
                nn.ConvTranspose2d(in_features, out_features, kernel_size=3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
        model += [nn.ReflectionPad2d(3), nn.Conv2d(64, channels, kernel_size=7), nn.Tanh()]
        self.model = nn.Sequential(*model)
    def forward(self, x):
        return self.model(x)

In [16]:
def enhance_yolo_validation_set():
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Running enhancement on: {device}")

    G_AB = GeneratorResNet().to(device)

    weights_path = "/content/drive/MyDrive/Night_Owl_Project/CycleGAN_Weights/G_AB_195.pth"
    G_AB.load_state_dict(torch.load(weights_path, map_location=device))
    G_AB.eval()


    input_img_dir = "/content/drive/MyDrive/yolo_dataset/images/val"
    input_lbl_dir = "/content/drive/MyDrive/yolo_dataset/labels/val"

    output_img_dir = "/content/drive/MyDrive/yolo_dataset/images/val_enhanced"
    output_lbl_dir = "/content/drive/MyDrive/yolo_dataset/labels/val_enhanced"

    os.makedirs(output_img_dir, exist_ok=True)
    os.makedirs(output_lbl_dir, exist_ok=True)

    # image transformations (normalization for the GAN)
    img_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    print("Enhancing validation images...")

    image_files = [f for f in os.listdir(input_img_dir) if f.endswith(('.jpg', '.png','.JPG',".JPEG"))]

    for count, img_name in enumerate(image_files):
        img_path = os.path.join(input_img_dir, img_name)

        original_img = Image.open(img_path).convert("RGB")
        orig_width, orig_height = original_img.size

        # CycleGANs require image dimensions to be multiples of 4
        # We temporarily resize to the nearest multiple of 4 to prevent tensor errors
        safe_width = (orig_width // 4) * 4
        safe_height = (orig_height // 4) * 4
        resized_img = original_img.resize((safe_width, safe_height), Image.BICUBIC)

        # Convert to tensor and pass through Generator
        img_tensor = img_transform(resized_img).unsqueeze(0).to(device)

        with torch.no_grad():
            fake_bright_tensor = G_AB(img_tensor)

        # Denormalize [-1, 1] back to [0, 1]
        fake_bright_tensor = (fake_bright_tensor.squeeze(0) + 1) / 2.0

        save_path = os.path.join(output_img_dir, img_name)
        save_image(fake_bright_tensor, save_path)

        txt_name = img_name.rsplit('.', 1)[0] + '.txt'
        src_txt = os.path.join(input_lbl_dir, txt_name)
        dst_txt = os.path.join(output_lbl_dir, txt_name)
        if os.path.exists(src_txt):
            shutil.copy(src_txt, dst_txt)

        if count % 100 == 0 and count > 0:
            print(f"Processed {count}/{len(image_files)} images...")

    print("\nenhancement complete")

if __name__ == '__main__':
    enhance_yolo_validation_set()

Running enhancement on: cuda
Enhancing validation images...
Processed 100/1913 images...
Processed 200/1913 images...
Processed 300/1913 images...
Processed 400/1913 images...
Processed 500/1913 images...
Processed 600/1913 images...
Processed 700/1913 images...
Processed 800/1913 images...
Processed 900/1913 images...
Processed 1000/1913 images...
Processed 1100/1913 images...
Processed 1200/1913 images...
Processed 1300/1913 images...
Processed 1400/1913 images...
Processed 1500/1913 images...
Processed 1600/1913 images...
Processed 1700/1913 images...
Processed 1800/1913 images...
Processed 1900/1913 images...

enhancement complete
